In [1]:
import pandas as pd
import gc

In [2]:
df2 = pd.read_pickle('../data/df2.pkl')

In [ ]:
df2.head()

In [ ]:
df2.info() # (49333714, 25)

In [11]:
# select representative date
df2 = df2[df2['ENTRY_DT'] == '2025-02-11']

In [ ]:
gc.collect() # free up storage immediately to delete overwritten files - the big df2 at the start
print(df2.shape) # (3846784, 25) for 11 feb

# Rule Based Classification

#### 1. Binary Criteria
- Drop rows with missing critical data
- The journey stage has no subsequent recorded trips on the same day and is therefore treated as a journey termination. It is the last ride of the day for the commuter.
    - Implementation: Sort by CRD_NUM and ENTRY_TM, flag rows where the next row belongs to a different CRD_NUM.
- 2 consecutive journey stages on the same bus service or train station indicates a return trip or intermediate activity. This should be classified as a new journey.
    - Implementation: Shift BUS_SVC_NUM and ORIG_LOC_ID_NUM down by 1 row to get the next stage's values, then flag rows where all 3 conditions are true simultaneously: same commuter, same bus service, and the alighting stop of stage 1 equals the boarding stop of stage 2.


In [12]:
# Drop rows with missing critical data
critical_cols = [
    'ENTRY_TM', 'EXIT_TM',
    'ORIG_LOC_ID_NUM', 'DEST_LOC_ID_NUM',
    'ORIG_LATITUDE', 'ORIG_LONGITUDE',
    'DEST_LATITUDE', 'DEST_LONGITUDE'
]

before = len(df2)
df2 = df2.dropna(subset=critical_cols).reset_index(drop=True)
after = len(df2)

print(f"Rows dropped: {before - after:,}")
print(f"Rows remaining: {after:,}")

Rows dropped: 29,725
Rows remaining: 3,817,059


In [13]:
# Binary criteria
# Sort first
df2 = df2.sort_values(['CRD_NUM', 'ENTRY_TM']).reset_index(drop=True)

# Shift to get next row's CRD_NUM
df2['next_CRD_NUM'] = df2['CRD_NUM'].shift(-1)

# Last stage = next row belongs to a different commuter
df2['is_last_stage'] = df2['CRD_NUM'] != df2['next_CRD_NUM'] # This tells us the last ride for the specific commuter

print(df2['is_last_stage'].value_counts())
print(f"\nUnique commuters: {df2['CRD_NUM'].nunique():,}") # Check that no. if True in is_last_stage == No. of unique commuters

is_last_stage
False    2606552
True     1210507
Name: count, dtype: int64

Unique commuters: 1,210,507


In [14]:
# Shift needed columns
df2['next_ORIG_LOC_ID_NUM'] = df2['ORIG_LOC_ID_NUM'].shift(-1)
df2['next_BUS_SVC_NUM']     = df2['BUS_SVC_NUM'].shift(-1)

# Flag same service return
df2['same_service_return'] = (
    (df2['CRD_NUM'] == df2['next_CRD_NUM']) &
    (df2['BUS_SVC_NUM'] == df2['next_BUS_SVC_NUM']) &
    (df2['DEST_LOC_ID_NUM'] == df2['next_ORIG_LOC_ID_NUM'])
)

print(df2['same_service_return'].value_counts())

same_service_return
False    3767169
True       49890
Name: count, dtype: int64


#### 2. Temporal criteria: 
- The binary criteria gives us pairs of consecutive stages that could be transfers. 
- Temporal criteria finds the time gap between alighting from stage 1 and boarding stage 2
- Implementation: 
    - gap = ENTRY_TM of stage 2 - EXIT_TM of stage 1. 
    - Shift ENTRY_TM and TRNSPT_MODE_CD down by 1 row to get the next stage's values. Compute the time gap between current EXIT_TM and next ENTRY_TM. Assign threshold (15 or 45 mins) based on transfer type, then flag if gap exceeds it.
    - Flag if gap > allowance.
- Allowance is given by **LTA's current window**:
    - Bus↔Bus or Bus↔Train transfers: 45 min maximum gap
    - Train↔Train transfers: 15 min maximum gap
    - Same bus service number: not a valid transfer (already handled in binary)
    - Max 5 transfers per journey
    - Overall journey duration (first to last boarding): max 2 hours


# PAUSE HERE BEFORE RUNNING BELOW. 

### Is this the correct temporal criteria?

* Note this is how TRNSPRT_MODE_CD is mapped.
1    BUS
2    RTS
3    BUS-RTS (interchange)

In [ ]:
# Shift needed columns 
df2['next_ENTRY_TM']       = df2['ENTRY_TM'].shift(-1)
df2['next_TRNSPT_MODE_CD'] = df2['TRNSPT_MODE_CD'].shift(-1)

#  Compute time gap
df2['time_gap_mins'] = (
    df2['next_ENTRY_TM'] - df2['EXIT_TM']
).dt.total_seconds() / 60

# Assign threshold by transfer type 
# RTS-RTS (2-2) → 15 mins, everything else → 45 mins
df2['time_threshold'] = df2.apply(lambda row:
    15 if (row['TRNSPT_MODE_CD'] == 2 and row['next_TRNSPT_MODE_CD'] == 2)
    else 45, axis=1
)

# Flag if gap exceeds threshold
df2['exceeds_time_allowance'] = (
    (df2['CRD_NUM'] == df2['next_CRD_NUM']) &
    (df2['time_gap_mins'] > df2['time_threshold'])
)

# Flag if overall journey exceeds 2 hours 
df2['journey_start_TM'] = df2.groupby('CRD_NUM')['ENTRY_TM'].transform('min')
df2['journey_duration_mins'] = (
    df2['ENTRY_TM'] - df2['journey_start_TM']
).dt.total_seconds() / 60
df2['exceeds_2hr'] = df2['journey_duration_mins'] > 120

# Flag if transfer count exceeds 5
df2['transfer_count'] = df2.groupby('CRD_NUM').cumcount()
df2['exceeds_5_transfers'] = df2['transfer_count'] > 5

# Combined temporal flag 
df2['temporal_new_journey'] = (
    df2['exceeds_time_allowance'] |
    df2['exceeds_2hr'] |
    df2['exceeds_5_transfers']
)

print(df2['exceeds_time_allowance'].value_counts())
print(df2['exceeds_2hr'].value_counts())
print(df2['exceeds_5_transfers'].value_counts())
print(f"\nTotal temporal new journey flags: {df2['temporal_new_journey'].sum():,}")

#### 3. Spatial Criteria:
- The spatial rule identifies whether the transition between consecutive journey stages is likely a transfer or the start of a new journey based on walking distance.
- Implementation: 
    - Use original clean df to access logtitude and latitude columns
    - Sort the dataset by card number (CRD_NUM) and timestamp (ENTRY_TM) to get consecutive stages for each rider.
    - For each ride, we look at the next stage’s boarding location (latitude and longitude). Rows without a next stage are ignored because there is nothing to compare. The person finished all their trips for the day. 
    - We create a boolean mask to select only rows that have a next stage (NEXT_ENTRY_LAT is not NaN). This helps us to ignore observations without next stages
    - Use Haversine formula to compute the shortest straight-line distance between DEST_LATITUDE & DEST_LONGTITUDE (alighting stop) and the NEXT_ENTRY_LONG & NEXT_ENTRY_LAT (the next entry point). Vectorisation is used because row-by-row computation was too slow. 
    - Allow a maximum transfer distance of 500 metres (we can change this)
    - If the distance exceeds a maximum allowed transfer distance (0.5 km), the row is flagged as spatial_new_journey = True. This indicates that the next ride is likely a new journey.

In [ ]:
df3 = pd.read_pickle('../data/df2.pkl')

In [ ]:
df3.head()

In [ ]:
df3 = df3.sort_values(by=['CRD_NUM', 'ENTRY_TM']).reset_index(drop=True)

In [ ]:
# Shift boarding coordinates to get "next stage" per rider
df3['NEXT_ENTRY_LAT'] = df3.groupby('CRD_NUM')['ORIG_LATITUDE'].shift(-1)
df3['NEXT_ENTRY_LON'] = df3.groupby('CRD_NUM')['ORIG_LONGITUDE'].shift(-1)

In [ ]:
import numpy as np
#use vectorized function so it runs much faster than row by row
def haversine_vectorized(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    
    r = 6371  # Earth radius in km
    return c * r

In [ ]:
#compute distance to next stage if it exists
mask = df3['NEXT_ENTRY_LAT'].notna()  # skip last stage (ie. if there is no next tap in) 
df3.loc[mask, 'distance_to_next_km'] = haversine_vectorized(
    df3.loc[mask, 'DEST_LATITUDE'].values,
    df3.loc[mask, 'DEST_LONGITUDE'].values,
    df3.loc[mask, 'NEXT_ENTRY_LAT'].values,
    df3.loc[mask, 'NEXT_ENTRY_LON'].values
)

In [ ]:
#flag spatial transfers
# set max reasonable walking transfer distance = 0.5 km 
# we can change this threshold
max_transfer_distance_km = 0.5

#initialise as False. Only mark as True if distance to next stage exceeds threshold
df3['spatial_new_journey'] = False
df3.loc[mask, 'spatial_new_journey'] = df3.loc[mask, 'distance_to_next_km'] > max_transfer_distance_km

In [ ]:
df3['spatial_new_journey'].value_counts()

# Internal Validity Check
- Drop rows with missing critical data
- Combine `pt_ride` and `pt_jrny` and only keep those where the ride entry time >= journey start time and ride exit time <= journey end time
- Generate confusion matrix calculate metrics (split rate, merge rate, sensitivity, specificity, accuracy)

In [2]:
df5 = pd.read_pickle('../data/df5.pkl')

In [3]:
df5.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33956202 entries, 0 to 33956201
Data columns (total 23 columns):
 #   Column              Dtype         
---  ------              -----         
 0   CRD_NUM             int64         
 1   JRNY_DEST_ID_NUM    float64       
 2   JRNY_START_DT       datetime64[ns]
 3   JRNY_START_TM       datetime64[ns]
 4   JRNY_END_DT         datetime64[ns]
 5   JRNY_END_TM         datetime64[ns]
 6   JRNY_ORIG_ID_NUM    float64       
 7   JRNY_DIST_KM_CNT    float64       
 8   JRNY_FARE_AMT       float64       
 9   JRNY_ID_NUM         int64         
 10  JRNY_TM_MIN_CNT     float64       
 11  PATRON_CATG_ID_NUM  int64         
 12  TRNSPT_MODE_CD      int64         
 13  DEST_STATION_NAME   object        
 14  DEST_MRK_ID_NUM     float64       
 15  DEST_LATITUDE       float64       
 16  DEST_LONGITUDE      float64       
 17  DEST_Travel_Type    object        
 18  ORIG_STATION_NAME   object        
 19  ORIG_MRK_ID_NUM     float64       
 20  

In [ ]:
# Drop rows with missing critical data
critical_cols = [
    'JRNY_START_TM', 'JRNY_END_TM',
    'JRNY_ORIG_ID_NUM', 'JRNY_DEST_ID_NUM',
    'ORIG_LATITUDE', 'ORIG_LONGITUDE',
    'DEST_LATITUDE', 'DEST_LONGITUDE'
]

before = len(df5)
df5 = df5.dropna(subset=critical_cols).reset_index(drop=True)
after = len(df5)

print(f"Rows dropped: {before - after:,}")
print(f"Rows remaining: {after:,}")

In [ ]:
df4 = df5[df5['JRNY_START_DT'] == '2025-02-11']

#### 1. Binary Criteria

In [ ]:
# Combine pt_ride and pt_jrny and only keep those where the ride entry time = 
df6 = df2.merge(df4, on=['CRD_NUM', 'JRNY_ID_NUM'], suffixes=('', '_JRNY'))
df6 = df6[(df6['ENTRY_DT'] >= df6['JRNY_START_DT']) & (df6['EXIT_DT'] <= df6['JRNY_END_DT'])]

In [ ]:
# Shift needed columns
df6['next_JRNY_ID_NUM'] = df6['JRNY_ID_NUM'].shift(-1)

# Flag true single journeys
df6['true_same_journey'] = (df6['JRNY_ID_NUM'] == df6['next_JRNY_ID_NUM'])

print(df6['true_same_journey'].value_counts())

In [ ]:
# Confusion matrix
pd.crosstab(
    df6['true_same_journey'],
    df6['same_service_return'],
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
total_true = (df6['true_same_journey'] == True).sum()

# Split Error Rate (FN)
split_error = ((df6['true_same_journey'] == True) & (df6['same_service_return'] == False)).sum()
split_rate = split_error / total_true
print("Split rate:", split_rate)

# Merge Error Rate (FP)
merge_error = ((df6['true_same_journey'] == False) & (df6['same_service_return'] == True)).sum()
merge_rate = merge_error / total_true
print("Merge rate:", merge_rate)

# Sensitivity (TP / (TP + FN))
tp = ((df6['true_same_journey'] == True) & (df6['same_service_return'] == True)).sum()
sensitivity = tp / (tp + split_error)
print("Sensitivity:", sensitivity)

# Specificity (TN / (TN + FP))
tn = ((df6['true_same_journey'] == False) & (df6['same_service_return'] == False)).sum()
specificity = tn / (tn + merge_error)
print("Specificity", specificity)

# Accuracy
accuracy = (tp + tn) / (total_true + merge_error + tn)
print("Accuracy:", accuracy)

#### 2. Temporal Criteria

In [ ]:
df6['true_new_journey'] = (df6['JRNY_ID_NUM'] != df6['next_JRNY_ID_NUM'])

In [ ]:
# Confusion matrix
pd.crosstab(
    df6['true_new_journey'],
    df6['temporal_new_journey'],
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
# Split Error Rate (FN)
split_error = ((df6['true_new_journey'] == True) & (df6['temporal_new_journey'] == False)).sum()
split_rate = split_error / total_true
print("Split rate:", split_rate)

# Merge Error Rate (FP)
merge_error = ((df6['true_new_journey'] == False) & (df6['temporal_new_journey'] == True)).sum()
merge_rate = merge_error / total_true
print("Merge rate:", merge_rate)

# Sensitivity (TP / (TP + FN))
tp = ((df6['true_new_journey'] == True) & (df6['temporal_new_journey'] == True)).sum()
sensitivity = tp / (tp + split_error)
print("Sensitivity:", sensitivity)

# Specificity (TN / (TN + FP))
tn = ((df6['true_new_journey'] == False) & (df6['temporal_new_journey'] == False)).sum()
specificity = tn / (tn + merge_error)
print("Specificity", specificity)

# Accuracy
accuracy = (tp + tn) / (total_true + merge_error + tn)
print("Accuracy:", accuracy)

#### 3. Spatial Criteria

In [ ]:
# Combine pt_ride and pt_jrny and only keep those where the ride entry time = 
df7 = df3.merge(df5, on=['CRD_NUM', 'JRNY_ID_NUM'], suffixes=('', '_JRNY'))
df7 = df7[(df7['ENTRY_DT'] >= df7['JRNY_START_DT']) & (df7['EXIT_DT'] <= df7['JRNY_END_DT'])]

In [ ]:
# Shift needed columns
df7['next_JRNY_ID_NUM'] = df7['JRNY_ID_NUM'].shift(-1)

# Flag true single journeys
df7['true_same_journey'] = (df7['JRNY_ID_NUM'] == df7['next_JRNY_ID_NUM'])

print(df7['true_same_journey'].value_counts())

In [ ]:
# Confusion matrix
pd.crosstab(
    df7['true_new_journey'],
    df7['spatial_new_journey'],
    rownames=['Actual'],
    colnames=['Predicted']
)

In [ ]:
total_true2 = (df7['true_same_journey'] == True).sum()

# Split Error Rate (FN)
split_error = ((df7['true_new_journey'] == True) & (df7['spatial_new_journey'] == False)).sum()
split_rate = split_error / total_true2
print("Split rate:", split_rate)

# Merge Error Rate (FP)
merge_error = ((df7['true_new_journey'] == False) & (df7['spatial_new_journey'] == True)).sum()
merge_rate = merge_error / total_true2
print("Merge rate:", merge_rate)

# Sensitivity (TP / (TP + FN))
tp = ((df7['true_new_journey'] == True) & (df7['spatial_new_journey'] == True)).sum()
sensitivity = tp / (tp + split_error)
print("Sensitivity:", sensitivity)

# Specificity (TN / (TN + FP))
tn = ((df7['true_new_journey'] == False) & (df7['spatial_new_journey'] == False)).sum()
specificity = tn / (tn + merge_error)
print("Specificity", specificity)

# Accuracy
accuracy = (tp + tn) / (total_true2 + merge_error + tn)
print("Accuracy:", accuracy)

# Calculating Walking distance

In [3]:
df2 = df2[df2['ENTRY_DT'] == '2025-02-11'].copy()
df3 = df2.sort_values(["CRD_NUM", "ENTRY_TM"], ascending= [True, True]).reset_index(drop= True)
df3.head(10)

,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,RIDE_DISC_AMT,...,DEST_STATION_NAME,DEST_MRK_ID_NUM,DEST_LATITUDE,DEST_LONGITUDE,DEST_Travel_Type,ORIG_STATION_NAME,ORIG_MRK_ID_NUM,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type
0,NaN,100005879220,22,2025-02-11,2025-02-11 11:20:34,2025-02-11,2025-02-11 11:32:25,110710909992,44,0.00,...,Yishun,22.0,1.429443,103.835005,TRAIN,Admiralty,44.0,1.440589,103.800990,TRAIN
1,NaN,100005879220,44,2025-02-11,2025-02-11 22:32:59,2025-02-11,2025-02-11 22:45:03,110710523649,22,0.00,...,Admiralty,44.0,1.440589,103.800990,TRAIN,Yishun,22.0,1.429443,103.835005,TRAIN
2,45.0,100006223599,3968,2025-02-11,2025-02-11 06:56:13,2025-02-11,2025-02-11 07:07:33,110710081977,4009,0.00,...,S'goon Stn Exit A/Blk 413,3968.0,1.349051,103.873414,BUS,Blk 172,4009.0,1.350814,103.889844,BUS
3,NaN,100006223599,109,2025-02-11,2025-02-11 07:09:17,2025-02-11,2025-02-11 07:18:09,110710081977,106,0.16,...,Boon Keng,109.0,1.319336,103.861570,TRAIN,Serangoon NEL,106.0,1.349708,103.873575,TRAIN
4,NaN,100006223599,106,2025-02-11,2025-02-11 14:51:22,2025-02-11,2025-02-11 15:03:53,110710904744,109,0.00,...,Serangoon NEL,106.0,1.349708,103.873575,TRAIN,Boon Keng,109.0,1.319336,103.861570,TRAIN
5,45.0,100006223599,4008,2025-02-11,2025-02-11 15:10:02,2025-02-11,2025-02-11 15:21:55,110710226201,3967,0.00,...,Opp Blk 169,4008.0,1.351105,103.890174,BUS,S'Goon Stn Exit B,3967.0,1.349098,103.873050,BUS
6,382.0,130013244516,5960,2025-02-11,2025-02-11 10:07:29,2025-02-11,2025-02-11 10:11:38,110710535706,5953,0.00,...,Opp One Punggol,5960.0,1.408865,103.906040,BUS,Blk 322 CP,5953.0,1.410690,103.897592,BUS
7,382.0,130013244516,5954,2025-02-11,2025-02-11 12:15:49,2025-02-11,2025-02-11 12:21:50,110709913132,5959,0.00,...,Opp Blk 322 CP,5954.0,1.410611,103.897717,BUS,One Punggol,5959.0,1.408195,103.905669,BUS
8,871.0,130013244638,2782,2025-02-11,2025-02-11 06:40:58,2025-02-11,2025-02-11 06:54:37,110711365561,6011,0.00,...,Bt Gombak Stadium,2782.0,1.357421,103.753093,BUS,Tengah CC,6011.0,1.356470,103.734420,BUS
9,NaN,130013244638,45,2025-02-11,2025-02-11 06:58:38,2025-02-11,2025-02-11 07:20:00,110711365561,29,0.50,...,Woodlands NSEW,45.0,1.436820,103.786067,TRAIN,Bukit Gombak,29.0,1.358612,103.751791,TRAIN


In [4]:
# Onemap API does not allow dates more than a year
df3["ENTRY_DT"] = pd.Timestamp("10-03-2026")
df3["EXIT_DT"] = pd.Timestamp("10-03-2026")


df3["ENTRY_TM"] = df3["ENTRY_TM"].dt.strftime("%H:%M:%S")
df3["EXIT_TM"] = df3["EXIT_TM"].dt.strftime("%H:%M:%S")
df3["ENTRY_DT"] = df3["ENTRY_DT"].dt.strftime("%m-%d-%Y")
df3["EXIT_DT"] = df3["EXIT_DT"].dt.strftime("%m-%d-%Y")

In [5]:
df3["next_orig_lat"] = df3.groupby("CRD_NUM")["ORIG_LATITUDE"].shift(-1)
df3["next_orig_lon"] = df3.groupby("CRD_NUM")["ORIG_LONGITUDE"].shift(-1)
df3["next_orig_station"] = df3.groupby("CRD_NUM")["ORIG_STATION_NAME"].shift(-1)


# Filter because cannot run so many at once. DO SUGGEST SOLUTIONS
df3 = df3[:12]
df3.head(13)

,BUS_SVC_NUM,CRD_NUM,DEST_LOC_ID_NUM,ENTRY_DT,ENTRY_TM,EXIT_DT,EXIT_TM,JRNY_ID_NUM,ORIG_LOC_ID_NUM,RIDE_DISC_AMT,...,DEST_LONGITUDE,DEST_Travel_Type,ORIG_STATION_NAME,ORIG_MRK_ID_NUM,ORIG_LATITUDE,ORIG_LONGITUDE,ORIG_Travel_Type,next_orig_lat,next_orig_lon,next_orig_station
0,NaN,100005879220,22,10-03-2026,11:20:34,10-03-2026,11:32:25,110710909992,44,0.00,...,103.835005,TRAIN,Admiralty,44.0,1.440589,103.800990,TRAIN,1.429443,103.835005,Yishun
1,NaN,100005879220,44,10-03-2026,22:32:59,10-03-2026,22:45:03,110710523649,22,0.00,...,103.800990,TRAIN,Yishun,22.0,1.429443,103.835005,TRAIN,NaN,NaN,NaN
2,45.0,100006223599,3968,10-03-2026,06:56:13,10-03-2026,07:07:33,110710081977,4009,0.00,...,103.873414,BUS,Blk 172,4009.0,1.350814,103.889844,BUS,1.349708,103.873575,Serangoon NEL
3,NaN,100006223599,109,10-03-2026,07:09:17,10-03-2026,07:18:09,110710081977,106,0.16,...,103.861570,TRAIN,Serangoon NEL,106.0,1.349708,103.873575,TRAIN,1.319336,103.861570,Boon Keng
4,NaN,100006223599,106,10-03-2026,14:51:22,10-03-2026,15:03:53,110710904744,109,0.00,...,103.873575,TRAIN,Boon Keng,109.0,1.319336,103.861570,TRAIN,1.349098,103.873050,S'Goon Stn Exit B
5,45.0,100006223599,4008,10-03-2026,15:10:02,10-03-2026,15:21:55,110710226201,3967,0.00,...,103.890174,BUS,S'Goon Stn Exit B,3967.0,1.349098,103.873050,BUS,NaN,NaN,NaN
6,382.0,130013244516,5960,10-03-2026,10:07:29,10-03-2026,10:11:38,110710535706,5953,0.00,...,103.906040,BUS,Blk 322 CP,5953.0,1.410690,103.897592,BUS,1.408195,103.905669,One Punggol
7,382.0,130013244516,5954,10-03-2026,12:15:49,10-03-2026,12:21:50,110709913132,5959,0.00,...,103.897717,BUS,One Punggol,5959.0,1.408195,103.905669,BUS,NaN,NaN,NaN
8,871.0,130013244638,2782,10-03-2026,06:40:58,10-03-2026,06:54:37,110711365561,6011,0.00,...,103.753093,BUS,Tengah CC,6011.0,1.356470,103.734420,BUS,1.358612,103.751791,Bukit Gombak
9,NaN,130013244638,45,10-03-2026,06:58:38,10-03-2026,07:20:00,110711365561,29,0.50,...,103.786067,TRAIN,Bukit Gombak,29.0,1.358612,103.751791,TRAIN,1.436867,103.786402,Woodlands Int B5


In [6]:
#To prevent repeated computations. (Method 1)


df3["pair_key"] = (
   df3["DEST_LATITUDE"].astype(str) + "_" +
   df3["DEST_LONGITUDE"].astype(str) + "_" +
   df3["next_orig_lat"].astype(str) + "_" +
   df3["next_orig_lon"].astype(str)
)


pairs = df3[
   ["pair_key", "DEST_LATITUDE", "DEST_LONGITUDE", "next_orig_lat", "next_orig_lon"]
].dropna(subset=["next_orig_lat", "next_orig_lon"]).drop_duplicates("pair_key").copy()

In [7]:
import requests
import numpy as np
import pandas as pd


TOKEN = ""


def get_walk_distance(row):
   if pd.isna(row["next_orig_lat"]) or pd.isna(row["next_orig_lon"]):
       return np.nan


   url = "https://www.onemap.gov.sg/api/public/routingsvc/route"


   params = {
       "start": f"{row['DEST_LATITUDE']},{row['DEST_LONGITUDE']}",
       "end": f"{row['next_orig_lat']},{row['next_orig_lon']}",
       "routeType": "walk"
   }


   headers = {"Authorization": TOKEN}


   r = requests.get(url, headers=headers, params=params)
   data = r.json()


   if data.get("status") == 0:
       return data["route_summary"]["total_distance"]
   else:
       return np.nan

In [8]:
pairs["walk_distance"] = pairs.apply(get_walk_distance, axis=1)

In [9]:
df3 = df3.merge(
   pairs[["pair_key", "walk_distance"]],
   on="pair_key",
   how="left"
)


In [10]:
df3[[
   "CRD_NUM",
   "DEST_LATITUDE",
   "DEST_LONGITUDE",
   "next_orig_lat",
   "next_orig_lon",
   'DEST_STATION_NAME',
   "next_orig_station",
   "walk_distance"
]].head(20)

,CRD_NUM,DEST_LATITUDE,DEST_LONGITUDE,next_orig_lat,next_orig_lon,DEST_STATION_NAME,next_orig_station,walk_distance
0,100005879220,1.429443,103.835005,1.429443,103.835005,Yishun,Yishun,NaN
1,100005879220,1.440589,103.800990,NaN,NaN,Admiralty,NaN,NaN
2,100006223599,1.349051,103.873414,1.349708,103.873575,S'goon Stn Exit A/Blk 413,Serangoon NEL,NaN
3,100006223599,1.319336,103.861570,1.319336,103.861570,Boon Keng,Boon Keng,NaN
4,100006223599,1.349708,103.873575,1.349098,103.873050,Serangoon NEL,S'Goon Stn Exit B,NaN
5,100006223599,1.351105,103.890174,NaN,NaN,Opp Blk 169,NaN,NaN
6,130013244516,1.408865,103.906040,1.408195,103.905669,Opp One Punggol,One Punggol,NaN
7,130013244516,1.410611,103.897717,NaN,NaN,Opp Blk 322 CP,NaN,NaN
8,130013244638,1.357421,103.753093,1.358612,103.751791,Bt Gombak Stadium,Bukit Gombak,NaN
9,130013244638,1.436820,103.786067,1.436867,103.786402,Woodlands NSEW,Woodlands Int B5,NaN
